# 進階:把城市變成 3D

前面貼了角色、按情景調了高度。這本把佔地擠成 3D 量體,三種方式看:matplotlib(頁面內、程式碼定角度)、plotly(瀏覽器滑鼠轉)、OBJ(匯出給 Rhino/Blender)。

## 怎麼執行

- 點一格,按 **Shift+Enter** 執行;或選單 Run → Run All Cells 從頭跑。
- 結果與圖會顯示在該格下面。
- 跟著說明,在標 ✏️ 的地方改設定,再重跑那一步。

In [ ]:
# 讓練習本找到 engine 裡的程式碼(這格不用改,直接執行)
import sys, os
for _p in (".", "engine", os.path.join("engine", "steps")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

## 準備

匯入工具、讓圖顯示在頁面,並取預設大巴窯建築、貼好角色標籤。

In [ ]:
import common, plots
import matplotlib; matplotlib.use("module://matplotlib_inline.backend_inline")
%matplotlib inline
df = common.assign_all(common.current_buildings())   # 載入並貼好角色標籤
print("建築數:", len(df), "|", df["stakeholder"].value_counts().to_dict())
print("本次輸出資料夾:", common.OUT.relative_to(common.ROOT), "(每次跑進各自時間戳夾,不覆寫)")
print(common.honest_note())

## 選用哪個高度

可用 current(原始觀測高),或某權力情景的高度。改 `HEIGHT_SCEN` 切換。

In [ ]:
HEIGHT_SCEN = "current"   # ✏️ 試 "developer_led"/"community_led"/"state_eco"/"developer_renewal"
scen = common.load_scenarios()
assert HEIGHT_SCEN in scen, "沒有這個情景,可選:%s" % list(scen.keys())
df["h3d"] = df["height_m"].values if HEIGHT_SCEN == "current" else common.scenario_heights(df, scen[HEIGHT_SCEN]).values
print("3D 用高度:", HEIGHT_SCEN, "| 平均 %.1f m 最高 %.1f m" % (df.h3d.mean(), df.h3d.max()))

## 取多少棟

為了畫得快,只取佔地最大的約 200 棟。改 `TOP_N` 調整,越大越慢。

In [ ]:
TOP_N = 200   # ✏️ 取佔地最大的幾棟;越大越慢
sub = df.sort_values("area_m2", ascending=False).head(TOP_N).copy()
print("3D 子集:%d 棟,佔總佔地 %.0f%%" % (len(sub), 100*sub.area_m2.sum()/df.area_m2.sum()))

## 方式 A:matplotlib

把每棟佔地按高度擠成盒子,在頁面顯示。角度由 elev(俯仰)、azim(旋轉)控制。

In [ ]:
plots.city_3d(sub, height_col="h3d", elev=30, azim=-60)   # ✏️ 改 elev/azim 換視角

## 方式 B:plotly

生成可在瀏覽器用滑鼠拖動旋轉的 3D,存成 html。轉到滿意角度後,用工具列相機鈕截圖。

In [ ]:
fig = plots.city_3d_plotly(sub, height_col="h3d")         # 可在瀏覽器拖動旋轉
out_html = common.OUT / ("city3d_%s.html" % HEIGHT_SCEN)
fig.write_html(str(out_html)); print("→ 寫好可旋轉 3D:", out_html.relative_to(common.ROOT))
fig.show()

## 方式 C:匯出 OBJ

用真實佔地擠成量體,匯出 .obj 給 Rhino 或 Blender 精確建模量測。

In [ ]:
obj, nv, nf = common.extrude_obj(sub, height_col="h3d")   # 匯出 OBJ 給 Rhino/Blender
out_obj = common.OUT / ("city_%s.obj" % HEIGHT_SCEN)
out_obj.write_text(obj, encoding="utf-8")
print("→ 匯出 OBJ:", out_obj.relative_to(common.ROOT), "| 頂點%d 面%d" % (nv, nf))

## 小結

- matplotlib:程式碼定角度,適合批量固定視角。
- plotly:滑鼠找角度,適合挑視角截圖。
- OBJ:交專業軟體,適合精確建模。